# Private P0 Colab: text, concat fusion, gated fusion
Select a GPU runtime and run sequentially. Keep the repository PRIVATE. Use a read-only private-repo token via hidden input, and authorized private Drive inputs. No raw audio or outside 2A audio is used. All jobs are pending until executed and validated. No secrets or cell outputs belong in Git.

Resume reuses completed folds with matching provenance; an interrupted fold restarts from its seed, not its saved epoch. Never run two sessions with the same RUN_ID. Main's `mer.training.p0_train` must be present in the chosen revision.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, hashlib, getpass, tempfile

drive.mount('/content/drive')
REPO = 'SandeshAwate6055/iiith-speech-a2'
REVISION = 'main'  # Prefer an immutable commit containing final P0 runner/canonical configs.
RUN_ID = 'p0_attempt_001'  # Reuse for continuation only; new ID when inputs/code/config change.
INPUT = Path('/content/drive/MyDrive/iiith_a2_inputs')
META = INPUT / 'metadata.csv'
SPLITS = INPUT / 'splits'
FEATURES = INPUT / 'features'  # Contains wavlm_base_plus/<utt_id>.pt
OUTPUT = Path('/content/drive/MyDrive/iiith_a2_runs') / RUN_ID
CHECKOUT = Path('/content/iiith-speech-a2-private')
assert RUN_ID and '/' not in RUN_ID and RUN_ID not in ('.','..')
assert not CHECKOUT.exists(), 'Existing checkout: inspect it or reconnect a clean runtime.'


In [ ]:
# Prompt is hidden; no token in URL, notebook source, git config or saved credential file.
token = getpass.getpass('Private GitHub repository read token: ')
assert token, 'Private-repo authentication required; do not make the repo public.'
with tempfile.TemporaryDirectory() as td:
    askpass = Path(td) / 'askpass.py'
    askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["PRIVATE_REPO_TOKEN"])\n')
    askpass.chmod(0o700)
    env = dict(os.environ, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', PRIVATE_REPO_TOKEN=token)
    try:
        cloned = subprocess.run(['git','-c','credential.helper=','clone','https://github.com/'+REPO+'.git',str(CHECKOUT)],env=env,capture_output=True)
    finally:
        env.pop('PRIVATE_REPO_TOKEN',None)
        del token
assert cloned.returncode == 0, 'Private clone failed: check repository/token access; no visibility change needed.'
subprocess.run(['git','checkout',REVISION],cwd=CHECKOUT,check=True,capture_output=True)
COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=CHECKOUT,text=True).strip()
PROJECT = CHECKOUT / 'assignment_2b_emotion'
os.chdir(PROJECT)
assert (PROJECT/'src/mer/training/p0_train.py').is_file(), 'Chosen revision lacks the P0 runner; stop.'
print('Revision:',COMMIT)


In [ ]:
# Install before importing torch/transformers. Restart runtime if Colab requests it;
# after restart retain/inspect checkout and re-establish variables without cloning over it.
subprocess.run([sys.executable,'-m','pip','install','torch==2.5.1','torchaudio==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')],check=True)
import torch
assert torch.cuda.is_available(), 'GPU required: change runtime type and reconnect.'
print(torch.__version__,torch.cuda.get_device_name(0))
os.environ['PYTHONPATH'] = str(PROJECT/'src')
sys.path.insert(0,str(PROJECT/'src'))
# Keep pretrained download cache private and outside Git.
os.environ['HF_HOME'] = str(OUTPUT.parent/'hf_cache')
subprocess.run([sys.executable,'-m','mer.training.p0_train','--help'],check=True)


In [ ]:
import pandas as pd
import yaml
assert META.is_file(), f'Missing metadata: {META}'
assert SPLITS.is_dir() and FEATURES.is_dir(), 'Missing Drive split/feature directories'
assert not OUTPUT.resolve().is_relative_to(CHECKOUT.resolve()), 'Outputs must be outside Git'
df=pd.read_csv(META)
required=['utt_id','emotion_id','transcript','speaker','word']
assert set(required)<=set(df.columns)
assert len(df)==2800 and df.utt_id.is_unique and not df[required].isna().any().any()
assert set(df.emotion_id)==set(range(7)) and (df.groupby('emotion_id').size()==400).all()
assert df.word.nunique()==200 and df.speaker.nunique()==2
ids=set(df.utt_id)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
hashes={'metadata':sha(META)}
protocols={}
for p in 'ABC':
    path=SPLITS/f'protocol_{p}.json'
    frozen=PROJECT/'splits'/path.name
    assert path.is_file() and frozen.is_file(), f'Missing split {p}'
    assert sha(path)==sha(frozen), f'Split {p} differs from frozen revision'
    hashes[p]=sha(path); protocols[p]=json.loads(path.read_text())
    for fold in protocols[p]['folds']:
        groups=[set(fold[x]) for x in ('train','val','test')]
        assert all(g and g<=ids for g in groups)
        assert all(not groups[i]&groups[j] for i,j in ((0,1),(0,2),(1,2)))
# Read every trusted tensor: fail before any job if cache is missing, malformed or nonfinite.
cache_hash=hashlib.sha256()
for uid in sorted(ids):
    assert Path(uid).name==uid and '/' not in uid and '\\' not in uid
    path=FEATURES/'wavlm_base_plus'/f'{uid}.pt'
    assert path.is_file(), f'Missing cache: {path}'
    tensor=torch.load(path,map_location='cpu',weights_only=True)
    assert tensor.ndim==3 and tensor.shape[0]==13 and tensor.shape[1]>0 and tensor.shape[2]==768, str(path)
    assert torch.isfinite(tensor).all(), f'Nonfinite cache: {path}'
    cache_hash.update((uid+sha(path)).encode())
hashes['features']=cache_hash.hexdigest()
print('Input preflight passed; all 2800 trusted WavLM caches checked')


In [ ]:
from copy import deepcopy
configs={}
for key in ('text','fusion_concat','fusion'):
    path=PROJECT/'configs'/f'{key}.yaml'
    assert path.is_file(), f'Missing config {key}'
    cfg=yaml.safe_load(path.read_text())
    textcfg=cfg['model'] if key=='text' else cfg['model']['text_cfg']
    assert textcfg['model_name']=='FacebookAI/roberta-base', 'Use final canonical RoBERTa configs'
    assert not cfg['data'].get('align_spans'), 'Cached P0 workflow must not depend on 2A audio'
    cfg['data'].update(meta=str(META),splits_dir=str(SPLITS),features_dir=str(FEATURES))
    cfg['ckpt_dir']=str(OUTPUT/'checkpoints'); cfg['out_dir']=str(OUTPUT/'results')
    configs[key]=cfg
spec={'commit':COMMIT,'inputs':hashes,'configs':configs}
fingerprint=hashlib.sha256(json.dumps(spec,sort_keys=True).encode()).hexdigest()
manifest=OUTPUT/'notebook_manifest.json'
if OUTPUT.exists():
    assert manifest.is_file(), 'Unowned existing output directory; use a new RUN_ID'
    assert json.loads(manifest.read_text())['fingerprint']==fingerprint, 'Changed inputs/code/config: use a new RUN_ID'
else:
    OUTPUT.mkdir(parents=True,exist_ok=False)
    manifest.write_text(json.dumps({'fingerprint':fingerprint,'spec':spec},indent=2))
config_dir=OUTPUT/'configs'; config_dir.mkdir(exist_ok=True)
for key,cfg in configs.items():
    path=config_dir/f'{key}.yaml'
    content=yaml.safe_dump(cfg)
    if path.exists(): assert path.read_text()==content, 'Refusing to overwrite a changed config'
    else: path.write_text(content)
env_path=OUTPUT/'environment.txt'
if not env_path.exists():
    env_path.write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Private output root:',OUTPUT)


In [ ]:
# Existing main-owned P0 CLI: no shared model/trainer modifications.
from mer.evaluation.validate_evaluation import validate_run
for key in ('text','fusion_concat','fusion'):
    mode='text' if key=='text' else 'fusion'
    for p in 'ABC':
        cfg=configs[key]; name=f"{cfg['name']}_proto{p}"
        out=OUTPUT/'results'/name
        ckpt=OUTPUT/'checkpoints'/name
        done=OUTPUT/f'{key}_proto{p}_DONE.json'
        expected=['summary.json','predictions.csv','embeddings.npy','train_history.json','p0_provenance.json']
        if done.exists():
            assert json.loads(done.read_text())['fingerprint']==fingerprint
            assert all((out/x).is_file() for x in expected), 'Completed job has missing outputs'
            assert all((ckpt/f"fold_{f['fold']}.pt").is_file() for f in protocols[p]['folds'])
            validate_run(out,meta_path=META,splits_dir=SPLITS)
            print('Skipping completed',name)
            continue
        # Main runner skips test if summary exists: reject an interrupted/partial evaluation.
        if (out/'summary.json').exists():
            assert all((out/x).is_file() for x in expected), 'Partial evaluation: preserve artifacts and use a fresh RUN_ID'
        print('Running/resuming',name,flush=True)
        with (OUTPUT/f'{key}_proto{p}.log').open('a') as log:
            subprocess.run([sys.executable,'-u','-m','mer.training.p0_train','--mode',mode,'--config',str(config_dir/f'{key}.yaml'),'--protocol',p],stdout=log,stderr=subprocess.STDOUT,check=True)
        assert all((out/x).is_file() for x in expected), f'Missing job artifacts: {out}'
        summary=json.loads((out/'summary.json').read_text())
        assert len(summary['folds'])==len(protocols[p]['folds'])
        for f in protocols[p]['folds']:
            assert (ckpt/f"fold_{f['fold']}.pt").is_file()
            assert (out/f"fold_{f['fold']}_complete.json").is_file()
        validate_run(out,meta_path=META,splits_dir=SPLITS)
        done.write_text(json.dumps({'fingerprint':fingerprint,'name':name},indent=2))
print('All nine requested jobs completed or verified as previously completed')


In [ ]:
# Recompute metrics from predictions, checking exact frozen-fold membership and labels.
from mer.evaluation.validate_evaluation import validate_run
rows=[]
for key in ('text','fusion_concat','fusion'):
    for p in 'ABC':
        name=f"{configs[key]['name']}_proto{p}"
        assert (OUTPUT/f'{key}_proto{p}_DONE.json').is_file(), 'Jobs still pending'
        _,report=validate_run(OUTPUT/'results'/name,meta_path=META,splits_dir=SPLITS)
        rows.append({'run':name,'protocol':p,**{k:report[k] for k in ('accuracy_mean','accuracy_std','uar_mean','uar_std','macro_f1_mean','macro_f1_std')}})
table=pd.DataFrame(rows)
# New artifact only: do not overwrite a previously generated table silently.
target=OUTPUT/'validated_metrics.csv'
content=table.to_csv(index=False)
if target.exists(): assert target.read_text()==content, 'Saved table changed; inspect before replacing'
else: target.write_text(content)
print('Validated nine completed jobs; metrics saved as fractions on private Drive.')


## Outputs and interpretation
Drive `results/<config-name>_proto<protocol>/` contains all-fold summary metrics, predictions, embeddings, histories and provenance; checkpoints are under `checkpoints/`. Logs are private Drive files. Re-run configuration/preflight cells in a fresh session with the same revision, input paths and RUN_ID to continue. Completed folds are skipped; interrupted folds restart deterministically. No optimizer-state/epoch resume is claimed. Partial evaluation requires a new RUN_ID, preserving old artifacts.

To inspect without rerunning, read saved JSON/CSV directly. Metrics are fractions and population standard deviations over folds. Classical train+validation fitting differs from neural train-only fitting. Text C's existing 0% baseline reflects complementary imbalance, not semantic emotion inference; see README. No completed results are claimed by this unexecuted notebook. Keep outputs/caches/checkpoints private and clear all cell outputs before committing.